In [1]:
# set the project root and import the current CORD gate utilities
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.cord_risk.train_cord_review_gate import (
    FEATURE_COLUMNS,
    CordGateConfig,
    make_cord_training_table,
)

In [2]:
# build the current CORD training table exactly as the gate sees it
cord_df = make_cord_training_table(data_root=PROJECT_ROOT / "Data", split="train")

print(cord_df.shape)
display(cord_df.head())
print(sorted(cord_df.columns))

(800, 39)


,doc_id,dataset,split,n_tokens,n_boxes,menu_count,total_present,subtotal_present,tax_present,service_present,...,has_cash_anchor,has_change_anchor,total_math_gap,total_math_gap_ratio,total_math_check_available,token_box_ratio,amount_token_ratio,anchor_count,menu_token_ratio,risk_label
0,cord_train_0,CORD,train,135,135,22,True,True,True,True,...,False,False,45.0,0.000028,True,1.0,0.348148,2,0.162963,1
1,cord_train_1,CORD,train,39,39,6,True,True,True,True,...,False,False,0.0,0.000000,True,1.0,0.410256,2,0.153846,0
2,cord_train_2,CORD,train,39,39,5,True,True,False,False,...,True,True,0.0,0.000000,True,1.0,0.384615,4,0.128205,0
3,cord_train_3,CORD,train,22,22,3,True,True,True,True,...,False,False,19000.0,0.062911,True,1.0,0.500000,3,0.136364,0
4,cord_train_4,CORD,train,18,18,4,True,True,True,False,...,False,False,0.0,0.000000,True,1.0,0.500000,2,0.222222,0


['amount_token_ratio', 'anchor_count', 'cash_in_ocr', 'cash_present', 'change_in_ocr', 'change_present', 'dataset', 'doc_id', 'exact_subtotal_matches', 'exact_tax_matches', 'exact_total_matches', 'has_cash_anchor', 'has_change_anchor', 'has_subtotal_anchor', 'has_tax_anchor', 'has_total_anchor', 'menu_count', 'menu_token_ratio', 'n_amount_like_tokens', 'n_boxes', 'n_tokens', 'risk_label', 'service_in_ocr', 'service_len', 'service_present', 'split', 'subtotal_in_ocr', 'subtotal_len', 'subtotal_present', 'tax_in_ocr', 'tax_len', 'tax_present', 'token_box_ratio', 'total_in_ocr', 'total_len', 'total_math_check_available', 'total_math_gap', 'total_math_gap_ratio', 'total_present']


In [3]:
# inspect class balance and core numeric feature distributions before trusting the gate
display(
    cord_df["risk_label"].value_counts(dropna=False).sort_index().to_frame("count")
)

print("positive rate:", cord_df["risk_label"].mean())

display(
    cord_df[
        [
            "n_tokens",
            "n_boxes",
            "menu_count",
            "total_len",
            "subtotal_len",
            "tax_len",
            "service_len",
            "exact_total_matches",
            "exact_subtotal_matches",
            "exact_tax_matches",
            "n_amount_like_tokens",
            "token_box_ratio",
            "amount_token_ratio",
            "anchor_count",
            "menu_token_ratio",
            "total_math_gap_ratio",
        ]
    ].describe()
)

,count
risk_label,
0,517
1,283


positive rate: 0.35375


,n_tokens,n_boxes,menu_count,total_len,subtotal_len,tax_len,service_len,exact_total_matches,exact_subtotal_matches,exact_tax_matches,n_amount_like_tokens,token_box_ratio,amount_token_ratio,anchor_count,menu_token_ratio,total_math_gap_ratio
count,800.000000,800.000000,800.000000,800.000000,800.00000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.0,800.000000,800.000000,800.000000,800.000000
mean,24.208750,24.208750,3.722500,6.362500,4.42375,2.490000,0.700000,1.973750,1.336250,0.437500,10.012500,1.0,0.425679,2.708750,0.178102,350.290335
std,15.651402,15.651402,2.211132,1.592722,3.58647,3.076806,1.894594,1.142231,1.372239,0.518585,6.346471,0.0,0.093472,1.022365,0.083956,5560.244043
min,5.000000,5.000000,2.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.028571,0.000000
25%,15.000000,15.000000,2.000000,6.000000,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000,6.000000,1.0,0.380542,2.000000,0.113131,0.000000
50%,20.000000,20.000000,3.000000,6.000000,6.00000,0.000000,0.000000,2.000000,1.000000,0.000000,8.000000,1.0,0.428571,3.000000,0.150000,0.000000
75%,28.000000,28.000000,4.000000,7.000000,6.00000,5.000000,0.000000,3.000000,2.000000,1.000000,12.000000,1.0,0.500000,3.000000,0.230769,0.000000
max,135.000000,135.000000,22.000000,20.000000,30.00000,23.000000,8.000000,5.000000,5.000000,2.000000,48.000000,1.0,0.700000,5.000000,0.571429,90911.202049


In [4]:
# inspect how often each review signal fires so we can see what is really driving the CORD proxy label
signal_cols = [
    "total_present",
    "subtotal_present",
    "tax_present",
    "service_present",
    "cash_present",
    "change_present",
    "total_in_ocr",
    "subtotal_in_ocr",
    "tax_in_ocr",
    "service_in_ocr",
    "cash_in_ocr",
    "change_in_ocr",
    "has_total_anchor",
    "has_subtotal_anchor",
    "has_tax_anchor",
    "has_cash_anchor",
    "has_change_anchor",
    "total_math_check_available",
]

display(
    cord_df[signal_cols + ["risk_label"]]
    .mean(numeric_only=True)
    .to_frame("fraction_true")
)

display(
    cord_df.groupby("risk_label")[
        [
            "n_tokens",
            "menu_count",
            "n_amount_like_tokens",
            "exact_total_matches",
            "total_math_gap_ratio",
            "anchor_count",
        ]
    ].mean()
)

,fraction_true
total_present,0.97375
subtotal_present,0.66125
tax_present,0.44250
service_present,0.12250
cash_present,0.64625
change_present,0.62500
total_in_ocr,0.97250
subtotal_in_ocr,0.65250
tax_in_ocr,0.43625
service_in_ocr,0.12250


,n_tokens,menu_count,n_amount_like_tokens,exact_total_matches,total_math_gap_ratio,anchor_count
risk_label,,,,,,
0,26.955513,3.615087,11.000000,1.522244,2.125204,2.735010
1,19.190813,3.918728,8.208481,2.798587,986.337589,2.660777


In [5]:
# inspect how often each individual CORD review rule fires
rule_df = pd.DataFrame({
    "miss_total_in_ocr": cord_df["total_present"] & (~cord_df["total_in_ocr"]),
    "miss_subtotal_in_ocr": cord_df["subtotal_present"] & (~cord_df["subtotal_in_ocr"]),
    "miss_tax_in_ocr": cord_df["tax_present"] & (~cord_df["tax_in_ocr"]),
    "miss_service_in_ocr": cord_df["service_present"] & (~cord_df["service_in_ocr"]),
    "too_many_total_matches": cord_df["exact_total_matches"] > 2,
    "many_amount_tokens": cord_df["n_amount_like_tokens"] > 25,
    "missing_total_anchor": cord_df["total_present"] & (~cord_df["has_total_anchor"]),
    "missing_tax_anchor": cord_df["tax_present"] & (~cord_df["has_tax_anchor"]),
    "extreme_token_count": (cord_df["n_tokens"] < 20) | (cord_df["n_tokens"] > 220),
    "dense_menu_sparse_ocr": (cord_df["menu_count"] >= 10) & (cord_df["n_tokens"] < 60),
    "large_total_math_gap": cord_df["total_math_check_available"] & (cord_df["total_math_gap_ratio"] > 0.03),
    "risk_label": cord_df["risk_label"].astype(bool),
})

display(rule_df.mean().to_frame("fraction_true"))

,fraction_true
miss_total_in_ocr,0.00125
miss_subtotal_in_ocr,0.00875
miss_tax_in_ocr,0.00625
miss_service_in_ocr,0.00000
too_many_total_matches,0.27375
many_amount_tokens,0.03875
missing_total_anchor,0.03000
missing_tax_anchor,0.23250
extreme_token_count,0.48750
dense_menu_sparse_ocr,0.00125


In [6]:
# count how many review rules fire per receipt
rule_cols = [col for col in rule_df.columns if col != "risk_label"]
rule_df["n_rules_fired"] = rule_df[rule_cols].sum(axis=1)

display(rule_df["n_rules_fired"].value_counts().sort_index().to_frame("count"))
display(rule_df.groupby("risk_label")["n_rules_fired"].describe())

,count
n_rules_fired,
0,182
1,335
2,258
3,24
4,1


,count,mean,std,min,25%,50%,75%,max
risk_label,,,,,,,,
False,517.0,0.647969,0.478066,0.0,0.0,1.0,1.0,1.0
True,283.0,2.091873,0.301364,2.0,2.0,2.0,2.0,4.0


In [7]:
# inspect which rule combinations dominate the positive class
positive_rules = rule_df.loc[rule_df["risk_label"], rule_cols].copy()
top_rule_counts = positive_rules.sum().sort_values(ascending=False).to_frame("positive_count")
display(top_rule_counts)

,positive_count
extreme_token_count,242
too_many_total_matches,177
missing_tax_anchor,86
large_total_math_gap,44
missing_total_anchor,20
many_amount_tokens,15
miss_subtotal_in_ocr,4
miss_tax_in_ocr,3
miss_total_in_ocr,1
miss_service_in_ocr,0


In [8]:
# inspect a few extreme arithmetic outliers before trusting total_math_gap_ratio as-is
display(
    cord_df[
        [
            "doc_id",
            "n_tokens",
            "menu_count",
            "total_len",
            "subtotal_len",
            "tax_len",
            "service_len",
            "exact_total_matches",
            "n_amount_like_tokens",
            "total_math_gap",
            "total_math_gap_ratio",
            "total_math_check_available",
            "risk_label",
        ]
    ]
    .sort_values("total_math_gap_ratio", ascending=False)
    .head(20)
)

,doc_id,n_tokens,menu_count,total_len,subtotal_len,tax_len,service_len,exact_total_matches,n_amount_like_tokens,total_math_gap,total_math_gap_ratio,total_math_check_available,risk_label
603,cord_train_603,35,4,6,20,5,0,2,13,7.363807e+09,90911.202049,True,1
150,cord_train_150,22,3,6,20,5,0,2,9,2.000000e+09,90909.090909,True,1
351,cord_train_351,27,2,6,20,5,0,2,11,3.136300e+09,90907.246377,True,1
267,cord_train_267,16,4,6,6,0,0,3,8,4.095900e+04,999.000000,True,1
795,cord_train_795,17,4,6,6,5,0,1,9,4.795200e+04,999.000000,True,1
787,cord_train_787,93,10,7,7,0,0,2,26,2.458269e+05,999.000000,True,1
216,cord_train_216,14,3,6,6,5,0,2,7,2.297700e+04,999.000000,True,1
714,cord_train_714,18,4,5,5,0,0,4,8,7.992000e+03,999.000000,True,1
616,cord_train_616,21,2,6,6,5,0,1,11,5.674320e+04,999.000000,True,0
80,cord_train_80,19,4,6,6,5,0,1,7,1.634664e+04,908.196955,True,1


In [9]:
# inspect the exact token-count distribution to understand why extreme_token_count fires so often
display(cord_df["n_tokens"].describe())

display(
    pd.cut(
        cord_df["n_tokens"],
        bins=[0, 10, 15, 20, 30, 50, 100, 220, 1000],
        right=False,
    ).value_counts().sort_index().to_frame("count")
)

count    800.000000
mean      24.208750
std       15.651402
min        5.000000
25%       15.000000
50%       20.000000
75%       28.000000
max      135.000000
Name: n_tokens, dtype: float64

,count
n_tokens,
"[0, 10)",26
"[10, 15)",170
"[15, 20)",194
"[20, 30)",229
"[30, 50)",133
"[50, 100)",44
"[100, 220)",4
"[220, 1000)",0


In [10]:
# inspect exact total-match distribution to see whether >2 is too aggressive
display(cord_df["exact_total_matches"].value_counts().sort_index().to_frame("count"))

display(
    cord_df.groupby("exact_total_matches")["risk_label"]
    .mean()
    .to_frame("positive_rate")
)

,count
exact_total_matches,
0,34
1,293
2,254
3,114
4,89
5,16


,positive_rate
exact_total_matches,
0,0.058824
1,0.197952
2,0.181102
3,0.710526
4,0.910112
5,0.937500


In [11]:
# clip and log-transform the unstable arithmetic gap to see whether it becomes more interpretable
import numpy as np

cord_df["total_math_gap_ratio_clipped"] = cord_df["total_math_gap_ratio"].clip(upper=10)
cord_df["log_total_math_gap_ratio"] = np.log1p(cord_df["total_math_gap_ratio_clipped"])

display(
    cord_df[
        ["total_math_gap_ratio", "total_math_gap_ratio_clipped", "log_total_math_gap_ratio"]
    ].describe()
)

display(
    cord_df.groupby("risk_label")[
        ["total_math_gap_ratio_clipped", "log_total_math_gap_ratio"]
    ].mean()
)

,total_math_gap_ratio,total_math_gap_ratio_clipped,log_total_math_gap_ratio
count,800.000000,800.000000,800.000000
mean,350.290335,0.253903,0.073330
std,5560.244043,1.493753,0.370761
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,90911.202049,10.000000,2.397895


,total_math_gap_ratio_clipped,log_total_math_gap_ratio
risk_label,,
0,0.055925,0.019653
1,0.615579,0.171391


In [13]:
# define cleaner transformed arithmetic features for label redesign
import numpy as np

cord_df["extreme_token_count_v2"] = (cord_df["n_tokens"] < 15) | (cord_df["n_tokens"] > 100)
cord_df["too_many_total_matches_v2"] = cord_df["exact_total_matches"] > 2
cord_df["missing_tax_anchor_v2"] = cord_df["tax_present"] & (~cord_df["has_tax_anchor"])
cord_df["missing_total_anchor_v2"] = cord_df["total_present"] & (~cord_df["has_total_anchor"])
cord_df["large_total_math_gap_v2"] = (
    cord_df["total_math_check_available"] &
    (cord_df["total_math_gap_ratio_clipped"] > 0.10)
)

display(
    cord_df[
        [
            "extreme_token_count_v2",
            "too_many_total_matches_v2",
            "missing_tax_anchor_v2",
            "missing_total_anchor_v2",
            "large_total_math_gap_v2",
        ]
    ].mean().to_frame("fraction_true")
)

,fraction_true
extreme_token_count_v2,0.25000
too_many_total_matches_v2,0.27375
missing_tax_anchor_v2,0.23250
missing_total_anchor_v2,0.03000
large_total_math_gap_v2,0.05625


In [14]:
# build a revised CORD review label with less dependence on broad token-count firing
cord_df["risk_label_v2"] = (
    cord_df[
        [
            "extreme_token_count_v2",
            "too_many_total_matches_v2",
            "missing_tax_anchor_v2",
            "missing_total_anchor_v2",
            "large_total_math_gap_v2",
        ]
    ].sum(axis=1) >= 2
).astype(int)

display(
    pd.DataFrame({
        "current_risk_label_rate": [cord_df["risk_label"].mean()],
        "revised_risk_label_v2_rate": [cord_df["risk_label_v2"].mean()],
    })
)

display(cord_df["risk_label_v2"].value_counts().sort_index().to_frame("count"))

,current_risk_label_rate,revised_risk_label_v2_rate
0,0.35375,0.20125


,count
risk_label_v2,
0,639
1,161


In [15]:
# compare old vs revised label against the main structural signals
display(
    cord_df.groupby("risk_label_v2")[
        [
            "n_tokens",
            "menu_count",
            "n_amount_like_tokens",
            "exact_total_matches",
            "total_math_gap_ratio_clipped",
            "log_total_math_gap_ratio",
            "anchor_count",
        ]
    ].mean()
)

,n_tokens,menu_count,n_amount_like_tokens,exact_total_matches,total_math_gap_ratio_clipped,log_total_math_gap_ratio,anchor_count
risk_label_v2,,,,,,,
0,26.048513,3.666667,10.688576,1.741784,0.129315,0.039123,2.735524
1,16.906832,3.944099,7.329193,2.894410,0.748383,0.209096,2.602484


In [16]:
# inspect how many v2 rules fire per receipt
v2_rule_cols = [
    "extreme_token_count_v2",
    "too_many_total_matches_v2",
    "missing_tax_anchor_v2",
    "missing_total_anchor_v2",
    "large_total_math_gap_v2",
]

cord_df["n_v2_rules_fired"] = cord_df[v2_rule_cols].sum(axis=1)

display(cord_df["n_v2_rules_fired"].value_counts().sort_index().to_frame("count"))
display(cord_df.groupby("risk_label_v2")["n_v2_rules_fired"].describe())

,count
n_v2_rules_fired,
0,298
1,341
2,150
3,11


,count,mean,std,min,25%,50%,75%,max
risk_label_v2,,,,,,,,
0,639.0,0.533646,0.499257,0.0,0.0,1.0,1.0,1.0
1,161.0,2.068323,0.253087,2.0,2.0,2.0,2.0,3.0


In [17]:
# see which v2 rules dominate the revised positive class
display(
    cord_df.loc[cord_df["risk_label_v2"] == 1, v2_rule_cols]
    .mean()
    .sort_values(ascending=False)
    .to_frame("fraction_of_v2_positives")
)

,fraction_of_v2_positives
extreme_token_count_v2,0.838509
too_many_total_matches_v2,0.652174
missing_tax_anchor_v2,0.304348
large_total_math_gap_v2,0.155280
missing_total_anchor_v2,0.118012


In [18]:
# compare old and revised labels directly
comparison_df = pd.DataFrame({
    "risk_label": cord_df["risk_label"],
    "risk_label_v2": cord_df["risk_label_v2"],
})

display(pd.crosstab(comparison_df["risk_label"], comparison_df["risk_label_v2"]))

risk_label_v2,0,1
risk_label,,
0,517,0
1,122,161


In [19]:
# quick baseline check for the revised label using individual v2 rules and the 2-rule threshold
baseline_rows = []

target = cord_df["risk_label_v2"].astype(bool)

for col in v2_rule_cols:
    pred = cord_df[col].astype(bool)

    tp = int((target & pred).sum())
    tn = int((~target & ~pred).sum())
    fp = int((~target & pred).sum())
    fn = int((target & ~pred).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    baseline_rows.append({
        "baseline": col,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pred_rate": pred.mean(),
    })

pred_two_rule = (cord_df[v2_rule_cols].sum(axis=1) >= 2)

tp = int((target & pred_two_rule).sum())
tn = int((~target & ~pred_two_rule).sum())
fp = int((~target & pred_two_rule).sum())
fn = int((target & ~pred_two_rule).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

baseline_rows.append({
    "baseline": "v2_two_or_more_rules",
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "pred_rate": pred_two_rule.mean(),
})

display(pd.DataFrame(baseline_rows).sort_values("f1", ascending=False))

,baseline,precision,recall,f1,pred_rate
5,v2_two_or_more_rules,1.000000,1.000000,1.000000,0.20125
0,extreme_token_count_v2,0.675000,0.838509,0.747922,0.25000
1,too_many_total_matches_v2,0.479452,0.652174,0.552632,0.27375
2,missing_tax_anchor_v2,0.263441,0.304348,0.282421,0.23250
4,large_total_math_gap_v2,0.555556,0.155280,0.242718,0.05625
3,missing_total_anchor_v2,0.791667,0.118012,0.205405,0.03000


In [20]:
# inspect positive rates conditional on each v2 rule
for col in v2_rule_cols:
    print("=" * 80)
    print(col)
    display(
        cord_df.groupby(col)["risk_label_v2"]
        .mean()
        .to_frame("positive_rate")
    )
    display(
        cord_df[col].value_counts(dropna=False).to_frame("count")
    )

extreme_token_count_v2


,positive_rate
extreme_token_count_v2,
False,0.043333
True,0.675000


,count
extreme_token_count_v2,
False,600
True,200


too_many_total_matches_v2


,positive_rate
too_many_total_matches_v2,
False,0.096386
True,0.479452


...

,positive_rate
missing_total_anchor_v2,
False,0.182990
True,0.791667


,count
missing_total_anchor_v2,
False,776
True,24


large_total_math_gap_v2


,positive_rate
large_total_math_gap_v2,
False,0.180132
True,0.555556


,count
large_total_math_gap_v2,
False,755
True,45


In [21]:
# see what happens if we remove the extreme-token rule from the v2 definition
cord_df["risk_label_v3_no_token_rule"] = (
    cord_df[
        [
            "too_many_total_matches_v2",
            "missing_tax_anchor_v2",
            "missing_total_anchor_v2",
            "large_total_math_gap_v2",
        ]
    ].sum(axis=1) >= 2
).astype(int)

display(
    pd.DataFrame({
        "risk_label_v2_rate": [cord_df["risk_label_v2"].mean()],
        "risk_label_v3_no_token_rule_rate": [cord_df["risk_label_v3_no_token_rule"].mean()],
    })
)

display(cord_df["risk_label_v3_no_token_rule"].value_counts().sort_index().to_frame("count"))

,risk_label_v2_rate,risk_label_v3_no_token_rule_rate
0,0.20125,0.04625


,count
risk_label_v3_no_token_rule,
0,763
1,37


In [22]:
# compare v2 vs v3 to see how much the token-count rule is driving the label
display(pd.crosstab(cord_df["risk_label_v2"], cord_df["risk_label_v3_no_token_rule"]))

display(
    cord_df.groupby("risk_label_v3_no_token_rule")[
        [
            "n_tokens",
            "menu_count",
            "n_amount_like_tokens",
            "exact_total_matches",
            "total_math_gap_ratio_clipped",
            "log_total_math_gap_ratio",
            "anchor_count",
        ]
    ].mean()
)

risk_label_v3_no_token_rule,0,1
risk_label_v2,,
0,639,0
1,124,37


,n_tokens,menu_count,n_amount_like_tokens,exact_total_matches,total_math_gap_ratio_clipped,log_total_math_gap_ratio,anchor_count
risk_label_v3_no_token_rule,,,,,,,
0,24.325033,3.714286,10.086501,1.954128,0.109939,0.033986,2.730013
1,21.810811,3.891892,8.486486,2.378378,3.222676,0.884662,2.270270
